# Latent gate: re-anchoring the answer at 1B, then Part C

Runs Amendment 1 of `results/latent_gate/heldout_scale/PLAN.md`. Results so far are in
`docs/LATENT_GATE_HELDOUT_RESULTS.md`.

- **B3, re-anchoring (about 1.5 h).**
  - Both LLaMA-3.2 1B models are scored with `dedup_anchor` and `anchor`.
  - These move later latent entries back to earlier positions and start the answer right after them. They target the
    position cost that Part B found.
  - **A download cell follows B3. Send that zip as soon as it arrives; Part C keeps running meanwhile.**
- **Part C (about 3–4.5 h).**
  - SIM-CoT-1B is fine-tuned three ways (`ft6`, `ft12`, `gate12_fixed`) and scored on validation.
  - An arm that passes the plan's rule is then scored on the held-out sets.
  - A second download follows.

**Before running:**
- **Runtime:** Runtime → Change runtime type → **A100 GPU**, then Runtime → **Run all**.
- **Drive space:** under 0.2 GB of new files. Part B's records must already be on Drive, under
  `latent-gate/heldout-scale/`. The fine-tuned 1B weights (about 5 GB each) stay on the Colab disk.
- **Resuming.** Evaluations resume from Drive. Trained weights don't survive a disconnect, so an arm whose validation
  scores are unfinished is retrained.
- **Keep the tab open.** If the browser asks whether this site may download multiple files, allow it.

In [ ]:
# Settings. Rerunning with the same settings resumes; finished pieces are skipped.
ANCHOR_MODELS = ["codi-llama1b", "simcot-llama1b"]
RUN_PART_C = True
PART_C_MODEL = "simcot-llama1b"   # fixed by Amendment 1
REPO_REF = "main"                 # the branch or commit holding Amendment 1
BATCH = 32                        # Part B's batch, so its records pair exactly with these

In [ ]:
from pathlib import Path
import hashlib, json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
OUT = DRIVE / "latent-gate" / "heldout-scale"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = OUT / "commit-amendment1.txt"  # Parts A and B keep commit.txt; the amendment pins its own code
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
plan = Path("results/latent_gate/heldout_scale/PLAN.md")
recorded = Path("results/latent_gate/heldout_scale/PLAN.sha256").read_text().split()[0]
assert hashlib.sha256(plan.read_bytes()).hexdigest() == recorded, "PLAN.md does not match PLAN.sha256"
assert "## Amendment 1" in plan.read_text() and Path("scripts/latent_gate/partc_train.py").exists(), \
    f"{ref} predates Amendment 1; push it first"
PINNED.write_text(COMMIT)
print("Code:", COMMIT, f"(from {ref})")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"  # local disk: the 1B checkpoints stay off Drive
os.environ["TOKENIZERS_PARALLELISM"] = "false"
LOCAL = Path("/content/partc")  # fine-tuned 1B weights, on the Colab disk only
LOCAL.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=OUT / "logs")

run("scripts/setup_data.py", "--eval-only")
assert (DRIVE / "data/gsm_train.json").exists() or not RUN_PART_C, "Part C needs data/gsm_train.json on Drive (Stage 1's)"

SIZES = {"gsm8k_valid": 500, "gsm8k_test": 1319, "gsm_hard": 1319, "multiarith": 180, "svamp": 1000}
HELDOUT = ["gsm8k_test", "gsm_hard", "multiarith", "svamp"]
ANCHOR_POLICIES = ["dedup_anchor", "anchor"]

def finished(folder, dataset):
    path = folder / f"{dataset}.jsonl"
    return path.exists() and len(path.read_text().splitlines()) == SIZES[dataset]

def evaluate(name, source, datasets, policies, *extra):
    """Score `source` (["--llama", x] plus optional ["--weights", dir]) into OUT/name, skipping finished datasets."""
    folder = OUT / name
    todo = [d for d in datasets if not finished(folder, d)]
    if not todo:
        print(f"{name}: {', '.join(datasets)} already on Drive")
        return
    run("scripts/latent_gate/heldout_eval.py", *source, "--datasets", *todo, "--output-dir", folder,
        "--policies", *policies, "--batch", BATCH, "--device", "cuda", *extra)

for model in ANCHOR_MODELS + ([PART_C_MODEL] if RUN_PART_C else []):
    missing = [d for d in ["gsm8k_valid"] + HELDOUT if not finished(OUT / model, d)]
    assert not missing, f"Part B's records for {model} are missing on Drive: {missing}"

## B3: re-anchoring at 1B

1. **Check 4.** On each model's real weights, moving every key and the query 18 positions later must leave the logits
   unchanged. It stops the notebook if it fails.
2. **Scoring.** `dedup_anchor` and `anchor`, at 12 and 24 steps, on validation and then the held-out sets.
3. **The report.** It pairs these with Part B's `plain` records.

In [ ]:
for model in ANCHOR_MODELS:
    marker = OUT / f"{model}-anchor" / "checks.txt"
    if not marker.exists():
        run("scripts/latent_gate/check_llama.py", "--llama", model, "--rotation", "--device", "cuda")  # raises if it fails
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text("E5 check 4 passed\n")
    evaluate(f"{model}-anchor", ["--llama", model], ["gsm8k_valid"], ANCHOR_POLICIES, "--depths", 12, 24, "--attention")
    evaluate(f"{model}-anchor", ["--llama", model], HELDOUT, ANCHOR_POLICIES, "--depths", 12, 24)
run("scripts/latent_gate/amendment_report.py", OUT)

In [ ]:
# Download 1: B3's evaluations and report. Send this zip as soon as it arrives; Part C runs on below.
from google.colab import files

def bundle(name, folders, report_args=()):
    target = Path("/content") / name
    shutil.rmtree(target, ignore_errors=True)
    target.mkdir()
    for folder in folders:
        if (OUT / folder).exists():
            shutil.copytree(OUT / folder, target / folder)
    report = subprocess.run([PYTHON, "scripts/latent_gate/amendment_report.py", str(OUT), *report_args],
                            capture_output=True, text=True, cwd=REPO_DIR)
    (target / "amendment_report.txt").write_text(report.stdout + report.stderr)
    (target / "commit.txt").write_text(COMMIT)
    archive = shutil.make_archive(str(target), "zip", target)
    files.download(archive)
    print(f"Downloading {archive}. If it does not start, get it from the Files panel on the left.")

bundle("latent-gate-anchor", [f"{m}-anchor" for m in ANCHOR_MODELS])

## Part C: fine-tuning SIM-CoT-1B (untriggered, exploratory)

1. **A two-update smoke run** of `gate12_fixed` checks memory, with micro-batches of 8, or of 4 if 8 does not fit. It also
   prints a time estimate.
2. **Training and scoring.** `ft6`, `ft12` and `gate12_fixed` are each trained (one pass, Stage 1's recipe) and scored on
   validation.
3. **The rule.** The report applies the plan's rule. Any arm that passes is scored on the held-out sets beside `ft6`.

In [ ]:
ARMS = {"ft6": ["plain"], "ft12": ["plain"], "gate12_fixed": ["plain", "gate", "gate_soft"]}
HELDOUT_POLICY = {"ft6": "plain", "ft12": "plain", "gate12_fixed": "gate"}
MICRO = None

def weights(arm):
    """The fine-tuned model directory on the Colab disk, training the arm first if it is not there."""
    global MICRO
    folder = LOCAL / arm
    if (folder / "model/pytorch_model.bin").exists():
        return folder / "model"
    if MICRO is None:
        for micro in (8, 4):
            try:
                run("scripts/latent_gate/partc_train.py", "--llama", PART_C_MODEL, "--arm", "gate12_fixed",
                    "--output", LOCAL / "smoke", "--micro-batch", micro, "--device", "cuda", "--bf16", "--smoke")
                MICRO = micro
                break
            except Exception as error:
                print(f"micro-batch {micro} did not run: {error}")
        assert MICRO, "Part C does not fit on this GPU even with micro-batches of 4"
    run("scripts/latent_gate/partc_train.py", "--llama", PART_C_MODEL, "--arm", arm, "--output", folder,
        "--micro-batch", MICRO, "--device", "cuda", "--bf16")
    logs = OUT / "arms" / f"{PART_C_MODEL}-{arm}"
    logs.mkdir(parents=True, exist_ok=True)
    for name in ("args.json", "metrics.jsonl"):
        shutil.copy2(folder / name, logs / name)
    if (folder / "model/gate.pt").exists():
        shutil.copy2(folder / "model/gate.pt", logs / "gate.pt")
    return folder / "model"

if RUN_PART_C:
    for arm, policies in ARMS.items():
        name = f"{PART_C_MODEL}-{arm}"
        if not finished(OUT / name, "gsm8k_valid"):
            evaluate(name, ["--llama", PART_C_MODEL, "--weights", weights(arm)], ["gsm8k_valid"], policies)
    verdict = OUT / "partc_verdict.json"
    run("scripts/latent_gate/amendment_report.py", OUT, "--verdict", verdict)
    passing = json.loads(verdict.read_text())["passing"]
    print("Arms passing the Part C rule:", passing or "none")
    for arm in (["ft6"] + passing) if passing else []:
        evaluate(f"{PART_C_MODEL}-{arm}", ["--llama", PART_C_MODEL, "--weights", weights(arm)], HELDOUT,
                 [HELDOUT_POLICY[arm]])
    run("scripts/latent_gate/amendment_report.py", OUT)

In [ ]:
# Download 2: everything from the amendment (B3 again, Part C's evaluations, training logs and gates).
partc = [f"{PART_C_MODEL}-{arm}" for arm in ARMS] + [f"arms/{PART_C_MODEL}-{arm}" for arm in ARMS] if RUN_PART_C else []
bundle("latent-gate-amendment1", [f"{m}-anchor" for m in ANCHOR_MODELS] + partc)  # not arms/ itself: Part A's GPT-2 weights

**Reading the results**
- **B3.** R1 is the main rule. It asks whether re-anchored soft dedup at 24 steps beats no eviction and stays within 2
  points of the 6-step score. R2 asks the same of re-anchoring alone, and G2 whether either beats 6 steps outright.
- **Part C.** An arm passes if at 12 steps it beats `ft6` on the 4+ step problems at p < 0.025, without losing overall or
  dropping by 24 steps. A gain counts only if it holds on the held-out sets.

**Sharing:** send the first zip as soon as it arrives, and the second when the run ends.